# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path("data/cookie_cats.csv"),
    Path("cookie_cats.csv"),
    Path("data/cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?


In [2]:
print("userid 결측:", int(df.userid.isna().sum()))
print("userid 중복 행:", int(df.userid.duplicated().sum()))
assert df.userid.notna().all() and df.userid.is_unique
print("version 고유값:", sorted(df.version.unique()))
version_summary = df.groupby("version").agg(
    users=("userid", "nunique"), mean_rounds=("sum_gamerounds", "mean"),
    retention_1=("retention_1", "mean"), retention_7=("retention_7", "mean")
).reindex(["gate_30", "gate_40"])
display(version_summary)
print("유지율은 0~1 비율이며 전체 배정 사용자를 분모로 합니다.")


userid 결측: 0
userid 중복 행: 0
version 고유값: ['gate_30', 'gate_40']


,users,mean_rounds,retention_1,retention_7
version,,,,
gate_30,44700,52.456264,0.448188,0.190201
gate_40,45489,51.298776,0.442283,0.182000


유지율은 0~1 비율이며 전체 배정 사용자를 분모로 합니다.


### 필수 1 답변

- **랜덤화 단위:** `userid`로 식별되는 사용자입니다. 이 파일은 사용자 1명당 1행이지만 실제 배정 구현은 별도 로그로 확인해야 합니다.
- **집단:** `gate_30`을 기존 대조군, `gate_40`을 게이트 위치를 바꾼 실험군으로 둡니다.
- **지표:** 핵심은 장기 재방문을 나타내는 `retention_7`, 보조는 이용 강도를 나타내는 평균 `sum_gamerounds`, 가드레일은 초기 이탈 악화를 감시하는 `retention_1`로 정합니다. 평균 라운드는 극단값에 민감하므로 분포도 함께 점검하고, 가드레일 허용 악화 폭은 실행 전에 확정합니다.
- **해석:** 요약 차이는 표본 변동·배정 또는 수집 문제로도 생길 수 있습니다. 실험 무결성, 불확실성, 사전 판단 기준을 확인해야 효과와 배포 여부를 결정할 수 있습니다.

---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?


In [3]:
original_version = df.version.copy(deep=True)
rng = np.random.default_rng(42)
randomized_users = df[["userid"]].drop_duplicates().copy()
randomized_users["practice_group"] = rng.choice(["A", "B"], size=len(randomized_users), p=[0.5,0.5])
assert randomized_users.userid.is_unique
assert randomized_users.groupby("userid").practice_group.nunique().eq(1).all()
practice_counts = randomized_users.practice_group.value_counts().reindex(["A","B"], fill_value=0)
actual_counts = df.version.value_counts().reindex(["gate_30","gate_40"])
display(pd.DataFrame({"연습 인원": practice_counts, "비율": practice_counts/practice_counts.sum()}))
display(pd.DataFrame({"실제 인원": actual_counts, "비율": actual_counts/actual_counts.sum()}))
expected_counts = np.repeat(actual_counts.sum()/2, 2)
allocation_test = stats.chisquare(actual_counts.to_numpy(), f_exp=expected_counts)
print(f"배정 비율 적합도 검정: χ²={allocation_test.statistic:.6f}, df=1, p={allocation_test.pvalue:.6g}")
print("50:50 배정과의 불일치 증거 있음" if allocation_test.pvalue < 0.05 else "50:50 배정과의 불일치 증거 부족")
assert df.version.equals(original_version)


,연습 인원,비율
practice_group,,
A,44852,0.497311
B,45337,0.502689


,실제 인원,비율
version,,
gate_30,44700,0.495626
gate_40,45489,0.504374


배정 비율 적합도 검정: χ²=6.902405, df=1, p=0.00860799
50:50 배정과의 불일치 증거 있음


### 필수 2 답변

- **시드:** 연습 배정 결과를 재현하기 위해 고정합니다. 실제 서비스에서는 동일 사용자가 재방문해도 배정이 유지되어야 합니다.
- **중복 소속:** 고유 사용자 목록에서 한 번씩 배정하고 `userid.is_unique`와 사용자별 그룹 수가 1인지 확인했습니다. 원래 `version`은 보존했습니다.
- **실제 배정:** 대조군 44,700명, 실험군 45,489명이고 χ²=6.9024, p=0.0086입니다. 0.05에서 50:50 가설을 기각하므로 계획 비율과의 불일치 신호(SRM)가 있습니다. 실제 배정 설정, 대상자 제외·중복·이벤트 누락 및 그룹별 수집 차이를 조사해야 합니다. 이 검정만으로 오류 원인을 특정하거나 무작위 배정 실패를 확정할 수는 없습니다.
- **사후 지표:** `retention_1`, `retention_7`, `sum_gamerounds`는 처치의 영향을 받을 수 있는 배정 후 결과입니다. 사전 공변량 균형 검사에는 배정 전 기기·지역·유입경로 같은 정보가 필요하며 이 파일에는 제공되지 않습니다.

---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


In [4]:
# 효과 검정 없이 관측 요약만 다시 계산합니다.
plan_summary = df.groupby("version").agg(
    users=("userid", "nunique"), mean_rounds=("sum_gamerounds", "mean"),
    retention_1=("retention_1", "mean"), retention_7=("retention_7", "mean")
).reindex(["gate_30", "gate_40"])
display(plan_summary)
print("관측 차이 gate_40 - gate_30:")
print("7일 유지율 차이(%p):", 100*(plan_summary.loc["gate_40","retention_7"]-plan_summary.loc["gate_30","retention_7"]))
print("1일 유지율 차이(%p):", 100*(plan_summary.loc["gate_40","retention_1"]-plan_summary.loc["gate_30","retention_1"]))
print("효과에 대한 통계적 검정 전이므로 최종 배포 판단은 보류합니다.")


,users,mean_rounds,retention_1,retention_7
version,,,,
gate_30,44700,52.456264,0.448188,0.190201
gate_40,45489,51.298776,0.442283,0.182000


관측 차이 gate_40 - gate_30:
7일 유지율 차이(%p): -0.8201298315205913
1일 유지율 차이(%p): -0.5905169787341458
효과에 대한 통계적 검정 전이므로 최종 배포 판단은 보류합니다.


### 과제: A/B 테스트 계획서

1. **가설 설정:** 핵심 지표를 7일 유지율로 고정합니다. H0는 p40=p30, H1은 p40≠p30인 양측 가설로 두고, 개선 방향은 p40>p30으로 정의합니다.
2. **실험 설계:** 신규 적격 사용자를 userid 단위로 50:50 배정하고 재접속에도 배정을 유지합니다. 대조군 gate_30, 실험군 gate_40으로 게이트 위치 외 조건은 같게 유지합니다. 핵심 retention_7, 보조 평균 sum_gamerounds, 가드레일 retention_1을 지정하고 유의수준 0.05, 실질적 개선 기준 및 가드레일 허용 악화 폭을 사전에 문서화합니다.
3. **실험 실행:** ID 중복·결측, 버전 값 및 배정/노출 일치, 재방문 이벤트 수집, 관찰 종료 시점과 7일 관찰 완료 여부를 점검합니다. 배정 비율 이상을 감시하고 예정된 모집 종료 후 마지막 사용자의 7일 결과가 성숙할 때까지 기다립니다. 친구 초대·공유 계정·경쟁 기능을 통한 사용자 간 영향, 동시 프로모션 및 다른 실험과의 상호작용을 검토하여 SUTVA의 간섭 없음 및 처치 일관성 전제를 점검합니다.
4. **결과 분석:** 원래 배정 기준(ITT)으로 전체 적격 사용자 유지율과 차이 및 신뢰구간을 계산하고, 사전에 정한 두 비율 비교로 검정합니다. 보조 지표는 탐색적으로 해석하고 핵심 지표와 가드레일을 함께 보며, 반복 조회에 따른 임의 조기 종료를 하지 않습니다. 현재 강의에서는 지표 요약까지만 수행하고 효과 검정은 후속 단계로 남깁니다.

### 최종 결론 (7문장)

현재 파일에서 gate_30과 gate_40은 각각 44,700명과 45,489명으로 구성됩니다.
7일 유지율은 각각 19.02%와 18.20%로 실험군이 0.82%p 낮습니다.
1일 유지율은 각각 44.82%와 44.23%입니다.
평균 라운드 수는 각각 52.46와 51.30이지만 극단값 영향을 확인해야 합니다.
배정 비율 검정에서는 p=0.0086으로 50:50 계획과의 불일치 신호가 확인되어 배정·제외·수집 로그를 조사해야 하며, 이는 효과 검정과 다른 질문입니다.
지표 차이에 대한 통계적 검정과 신뢰구간 분석은 아직 수행하지 않았고 실제 배정 로그·관찰 기간·간섭 가능성도 추가 확인해야 합니다.
따라서 지금은 gate_40의 인과효과와 최종 배포 결정을 보류합니다.

---

## 실습 마무리

- 어떤 문제가 있었는가?
- 어떻게 개선했는가?
- 무엇을 근거로 개선되었다고 판단했는가?

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.

### 마무리 답변

1. 평균이나 유지율 차이만으로 효과를 단정하면 배정 오류·중복·관찰 기간 차이·사용자 간 간섭을 놓칠 수 있습니다.

2. 사용자 단위 배정과 사전 지표 및 가드레일을 명시하고, 배정 비율을 검정하며 데이터 품질과 SUTVA 점검을 계획에 포함했습니다.

3. 사용자 ID 중복 0건과 배정 비율 검정 p=0.0086를 확인했습니다. p<0.05이므로 계획 비율과의 불일치 원인을 추가 조사해야 합니다. ID 중복이 없다는 사실만으로 랜덤화 구현이나 간섭 부재가 증명되지는 않습니다. 관측 지표는 효과 검정 전이므로 배포를 보류합니다.
